# STAC API + COG (Kanopia) — R demo

Goal (for a non-technical audience):
- Use a **STAC API** to find the right satellite products.
- Use a **COG** URL to read only the small chunk of raster data we need (fast preview).

This notebook targets the Kanopia STAC API endpoint:
`https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/`

In [ ]:
STAC_API_URL <- "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac"
# Approximate bounding box for Quebec
bbox <- c(-79.76259, 45.00495, -57.10592, 62.58502)

# Example query: September 2024
start_date <- "2024-09-01T00:00:00Z"
end_date <- "2024-10-01T23:59:59Z"
datetime_range <- paste0(start_date, "/", end_date)

keyword <- "sbl"
limit <- 50

cat("Querying STAC:\n")
cat("  ", STAC_API_URL, "\n", sep="")
cat("  datetime:", datetime_range, "\n", sep="")
cat("  bbox:", paste(bbox, collapse=","), "\n", sep="")
cat("  keyword:", keyword, "\n", sep="")
cat("  limit:", limit, "\n", sep="")


In [ ]:
# Workshop mode: DO NOT auto-install.
# Participants will learn what they need to install.
AUTO_INSTALL <- FALSE

needed <- c("httr", "jsonlite", "stringr", "terra")
missing <- needed[!sapply(needed, requireNamespace, quietly = TRUE)]

if (length(missing) > 0) {
  cat("Missing R packages:", paste(missing, collapse = ", "), "\n")
  cat("Install them in R, then restart this notebook/kernel:\n")
  cat(
    "  install.packages(c(", paste(sprintf('"%s"', missing), collapse = ", "), "), repos='https://cloud.r-project.org')\n",
    sep = ""
  )
  stop("Missing dependencies. Install and re-run this notebook.")
} else {
  cat("All required packages are already installed.\n")
}

library(httr)
library(jsonlite)
library(stringr)
library(terra)


In [ ]:
# --- Try to reuse the shared utilities (if they load in your environment) ---
utils_loaded <- FALSE
try({
  source("../scripts/R/lefolab_common_stac_utils.R")
  utils_loaded <- TRUE
}, silent = TRUE)

if (utils_loaded) {
  cat("Shared utilities loaded successfully.\n")
} else {
  cat("Could not load shared utilities; using internal fallback functions.\n")
}

# Helper: return y when x is NULL/NA (used to avoid missing operators)
`%||%` <- function(x, y) if (is.null(x) || is.na(x)) y else x

search_collections_fallback <- function(stac_api_url, bbox, datetime_range, keyword, limit) {
  url <- paste0(stac_api_url, "/search")
  body <- list(datetime = datetime_range, bbox = bbox, limit = limit)
  res <- httr::POST(url, body = body, encode = "json")
  httr::stop_for_status(res)
  data <- httr::content(res, as = "text", encoding = "UTF-8")
  data <- jsonlite::fromJSON(data)
  features <- data$features
  if (is.null(features)) return(list())

  # Filter by keyword in collection name
  if (!is.null(keyword) && keyword != "") {
    collection_names <- sapply(features, function(x) if (!is.null(x$collection)) x$collection else "")
    features <- features[str_detect(tolower(collection_names), keyword)]
  }
  return(as.list(features))
}

# Use shared search if available
if (utils_loaded) {
  stac_api_url_for_r <- paste0(STAC_API_URL, "/#/")
  collections <- search_collections(
    stac_api_url = stac_api_url_for_r,
    bbox = bbox,
    datetime_range = datetime_range,
    keyword = keyword,
    limit = limit
  )
} else {
  collections <- search_collections_fallback(
    stac_api_url = STAC_API_URL,
    bbox = bbox,
    datetime_range = datetime_range,
    keyword = keyword,
    limit = limit
  )
}

cat("Collections/features found:", length(collections), "\n")


In [ ]:
# --- Filter assets that look like COGs ---
is_cog_asset <- function(asset_key, asset_info) {
  key <- tolower(asset_key %||% "")
  mime <- tolower(asset_info$type %||% "")
  is_cogish <- (str_detect(mime, "cloud-optimized", negate = FALSE) || str_detect(key, "cog", negate = FALSE))
  return(is_cogish && !str_detect(key, "lowres") && !str_detect(key, "overview"))
}

# NOTE: %||% is not guaranteed to exist if shared utilities didn't load.
if (!utils_loaded) {
  `%||%` <- function(x, y) if (is.null(x) || is.na(x)) y else x
}

candidates <- list()
idx <- 1
for (f in collections) {
  assets <- f$assets
  if (is.null(assets)) next
  item_id <- f$id %||% NA
  collection <- f$collection %||% NA
  asset_names <- names(assets)
  if (is.null(asset_names)) next

  for (asset_key in asset_names) {
    asset_info <- assets[[asset_key]]
    if (!is.null(asset_info) && is_cog_asset(asset_key, asset_info)) {
      href <- asset_info$href
      if (!is.null(href) && href != "") {
        candidates[[idx]] <- list(href = href, asset_key = asset_key, item_id = item_id, collection = collection)
        idx <- idx + 1
      }
    }
  }
}

cat("Candidate COG assets:", length(candidates), "\n")
if (length(candidates) > 0) {
  for (i in seq_len(min(8, length(candidates)))) {
    c <- candidates[[i]]
    cat(sprintf("[%d] %s | item_id=%s\n      %s\n", i-1, c$asset_key, c$item_id, c$href))
  }
} else {
  cat("No COG-like assets found. Try changing keyword/limit/bbox.\n")
}


## Pick an asset to preview

Your STAC query returned a list of candidate **COG** raster assets.

- Set `target_index` to the number you want (shown in the list above, starting at 0).
- Then re-run the preview cell to read a small thumbnail from that COG.

In [ ]:
# Choose which candidate to preview
target_index <- 0  # change to 1,2,3,... to preview a different asset

if (length(candidates) == 0) stop("No candidates to preview.")
target <- candidates[[target_index + 1]]

href <- target$href
vsi_url <- paste0("/vsicurl/", href)

cat("Previewing asset:\n")
cat("  asset_key:", target$asset_key, "\n", sep="")
cat("  item_id:", target$item_id, "\n", sep="")
cat("  href:", href, "\n", sep="")

r <- rast(vsi_url)

cat("\nCOG metadata (quick):\n")
print(crs(r))
print(ext(r))
print(res(r))
cat("nrow/ncol:", nrow(r), "/", ncol(r), "\n", sep="")
cat("nlyr:", nlyr(r), "\n", sep="")

# Crop around the center to a manageable window, then aggregate for a thumbnail
nc <- ncol(r)
nr <- nrow(r)
win_px <- 1024
win_x <- min(win_px, nc)
win_y <- min(win_px, nr)

e <- ext(r)
rx <- res(r)[1]
ry <- abs(res(r)[2])

cx <- (e$xmin + e$xmax) / 2
cy <- (e$ymin + e$ymax) / 2

xmin <- cx - (rx * win_x) / 2
xmax <- cx + (rx * win_x) / 2
ymin <- cy - (ry * win_y) / 2
ymax <- cy + (ry * win_y) / 2

r_crop <- crop(r, ext(xmin, xmax, ymin, ymax))

# Downsample to about ~256 pixels
fact <- max(1, ceiling(max(ncol(r_crop), nrow(r_crop)) / 256))
r_small <- terra::aggregate(r_crop, fact = fact, fun = mean, na.rm = TRUE)

plot(r_small[[1]], col = gray.colors(255), main = paste0("COG preview: ", target$asset_key))


### Next steps

- Change `keyword` (and re-run the STAC search) to find different collections.
- Pick another `target_index` to compare bands/products.
- For a real workshop, you can repeat this with a different `datetime_range` and `bbox`.